# Test synchronisation Docs → Grist
Notebook de test pour valider chaque étape avant la synchro complète.

In [7]:
import sys
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv('../.env')

True

## 1. Connexion au client Docs
Renseigne `DOCS_SESSION_ID` et `DOCS_CSRF_TOKEN` dans le fichier `.env` avant de lancer cette cellule.

In [8]:
import os
from docs_client import DocsClient

docs = DocsClient(
    base_url='https://docs.numerique.gouv.fr',
    session_id=os.environ.get('DOCS_SESSION_ID'),
    csrf_token=os.environ.get('DOCS_CSRF_TOKEN'),
)
print('DocsClient initialisé.')

DocsClient initialisé.


## 2. Récupération du tree
Remplace l'URL ci-dessous par celle de ton document racine.

In [9]:
ROOT_URL = 'https://docs.numerique.gouv.fr/docs/21bb0cdd-638f-4a50-8f3a-03616716771e/'

doc_id = DocsClient.extract_doc_id(ROOT_URL)
tree = docs.get_tree(doc_id)

print(f"Racine : {tree['title']}")
print(f"Enfants directs : {len(tree.get('children', []))}")

Racine : Cartographie Stratégique du Bruit - Guide Méthodologique
Enfants directs : 13


## 3. Aplatissement du tree en records Grist

In [11]:
from importlib import reload
import sync
reload(sync)
from sync import flatten_tree

records = flatten_tree(tree, docs, 'https://docs.numerique.gouv.fr', is_root=True)

print(f"\n{len(records)} chapitre(s) extrait(s).")

[racine] Cartographie Stratégique du Bruit - Guide Méthodologique
  [1] 🌟 Guide
    [1.1] 📜 Rapportage
    [1.2] 🛣️ GITT
    [1.3] 🗺️ CBS
    [1.4] 🔠 Glossaire
    [1.5] ℹ️ À propos du guide
  [2] 🛠️ Outils
    [2.1] 🔊 NoiseModelling
    [2.2] 💽 Bases de données
    [2.3] 🖥️ Serveur de calcul
  [3] 🏘️ ​🔴 Bâtiments
    [3.1] 🙎‍♂️ ​🔴 Affectation des populations
    [3.2] 🧑‍🔧 Implémentation - Bâti
  [4] 📍 Récepteurs
    [4.1] 👩‍👦 ​🔵 Décompte de population
    [4.2] 🎨 Visuel - isosurfaces
    [4.3] 🧑‍🔧 ​🔵 Implémentation - Récepteurs
  [5] 🚙 Routier
    [5.1] 🌐 ​🔴 Géométrie
    [5.2] 📊 ​🔴 Débits
    [5.3] ⏩ ​🔴 Vitesses
    [5.4] 🛣️ Revêtements
    [5.5] ❄️ Pneus neige
    [5.6] 🧑‍🔧 Implémentation - Route
  [6] 🚅 Ferroviaire
    [6.1] 🚇 ​🔴 Tramway, Métro
    [6.2] 📊 ​🔵 Débits
    [6.3] ⏩ ​🔵 Vitesses
    [6.4] 🛤️ Plateforme
    [6.5] 🧑‍🔧 Implémentation - Fer
  [7] 🧱 ​🔴 Écrans acoustiques
    [7.1] 🧑‍🔧 Implémentation - Écrans
  [8] 🌉 ​🔵 Ponts et tunnels
  [9] 🏞️ Occupation du sol
    [9.1] 🧑‍🔧

## 4. Aperçu des records

In [ ]:
import pandas as pd

df = pd.DataFrame([r['fields'] for r in records])
df[['numero', 'niveau', 'ordre', 'titre', 'url']].head(20)

## 5. Envoi vers Grist
⚠️ Lance cette cellule uniquement quand les records semblent corrects.

In [ ]:
from grist_client import GristClient

grist = GristClient()
result = grist.add_records('Chapitres', records)
print(f"Enregistrements ajoutés : {result}")